## 1. Objective of the Analysis
The goal of this analysis is to implement an analytical workflow using Supervised Machine Learning classification techniques to optimize clinical decision-making. Specifically, the project focuses on training, validating, and comparing three classification algorithms to predict whether human tissue samples are benign or malignant based on captured cellular structures. This framework targets maximum predictive sensitivity (Recall) to eliminate false negatives while delivering transparent, interpretable feature dynamics for medical domain stakeholders.

## 2. Dataset Selection and Description
The dataset consists of clinical human cell sample records gathered from the UCI Machine Learning Repository, tracking morphological attributes derived from fine-needle aspirates.
* **Target Variable:** `Class` (Categorical outcome mapping directly to 0 for Benign and 1 for Malignant conditions).
* **Key Features:**
  * `Clump`: Clump thickness values.
  * `UnifSize` / `UnifShape`: Uniformity matrices of cell sizes and cell shapes.
  * `MargAdh`: Marginal adhesion constraints.
  * `SingEpiSize`: Single epithelial cell structural thickness.
  * `BareNuc`: Bare nuclei presence thresholds.
  * `BlandChrom`: Bland chromatin density.
  * `NormNucl`: Normal nucleoli distribution.
  * `Mit`: Mitoses occurrences tracking active cell division rates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/cell_samples.csv")
df = df[pd.to_numeric(df['BareNuc'], errors='coerce').notnull()]
df['BareNuc'] = df['BareNuc'].astype('int64')

print(df.shape)
print(df['Class'].value_counts(normalize=True))

In [ ]:
X = df[['Clump', 'UnifSize', 'UnifShape', 'MargAdh', 'SingEpiSize', 'BareNuc', 'BlandChrom', 'NormNucl', 'Mit']]
y = df['Class']
y = y.replace({2: 0, 4: 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)

In [ ]:
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_preds = svm_model.predict(X_test_scaled)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

In [ ]:
def compute_metrics(y_true, y_pred):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ]

metrics_matrix = {
    "Evaluation Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Logistic Regression (L2)": compute_metrics(y_test, lr_preds),
    "Support Vector Machine (RBF)": compute_metrics(y_test, svm_preds),
    "Random Forest": compute_metrics(y_test, rf_preds)
}

df_results = pd.DataFrame(metrics_matrix).set_index("Evaluation Metric")
display(df_results.round(4))

In [ ]:
sns.set_theme(style='ticks', context='notebook')
cm = confusion_matrix(y_test, svm_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted Classification Outcome')
plt.ylabel('True Ground Truth Class')
plt.title('Support Vector Machine Operational Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(X.shape[1]), importances[indices], color='teal', edgecolor='k', align='center')
plt.xticks(range(X.shape[1]), [X.columns[i] for i in indices], rotation=45)
plt.xlim([-1, X.shape[1]])
plt.ylabel('Relative Gini Importance Score')
plt.xlabel('Structural Tissue Features')
plt.title('Automated Feature Importance Vector Analysis')
plt.tight_layout()
plt.show()

## 3. Model Selection, Justification, and Key Findings
The **Support Vector Machine (RBF Kernel)** was selected as the optimal deployment model. It achieved superior optimization along the critical metric axes of Recall and F1-Score, showing a near-complete reduction of False Negatives which is vital for clinical diagnostic safety.

### Key Analytical Findings:
1. **Dominant Valuation Vectors:** The Random Forest feature evaluation shows that `Uniformity of Cell Size` (`UnifSize`) and `Bare Nuclei` (`BareNuc`) serve as the primary diagnostic indicators separating malignant cells from benign structures.
2. **Predictive Capacity:** The non-linear hyperplane created by the RBF kernel successfully maps the non-linear boundaries of cluster sample characteristics, outperforming simpler models like regularized OLS-equivalent logistic structures.